### Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import time
from tabulate import tabulate
from astropy.coordinates import SkyCoord
from astropy.table import Table, vstack
import astropy.units as u
import numpy as np

### Options

In [ ]:
export_all = False
fits_path = '../output/asterisms-2025-07'
fov = 2*u.arcmin

### Load Asterisms

In [ ]:
fits_file = f"{fits_path}/asterisms-GNAO.fits"
asterisms = Table.read(fits_file, format='fits')
print('Number of asterisms:', len(asterisms))

### Load Targets

In [ ]:
def remove_columns_except(table, keep_cols):
    for col in list(table.colnames):
        if col not in keep_cols:
            table.remove_column(col)

In [ ]:
# Load targets
target_mode = 5.2
match target_mode:
    case 0: # Ad hoc single target
        ra = 150.23723
        dec = 2.33716
        targets_name = 'single_target'
        targets = Table()
        targets.add_column(Table.Column(name='id', data=[1]))
        targets.add_column(Table.Column(name='ra', data=[ra]))
        targets.add_column(Table.Column(name='dec', data=[dec]))
        col_formats = ["", ".5f", ".5f"]
    case 1: # 3D-HST Sample Targets
        targets_name = 'sample-targets'
        fits_file = f"{fits_path}/sample-targets.fits"
        targets = Table.read(fits_file, format='fits')
        col_formats = ["", ".0f", ".5f", ".5f", ".3f", ".1f", ".1f", ".1f"]
    case 2: # Clusters for Lamiya
        targets = Table.read("../data/girmos-sci/lamiya_clusters.csv", format='csv')
        targets_name = 'lamiya_clusters'
        targets.rename_column('ID', 'id')
        targets.rename_column('Cluster', 'name')
        targets.rename_column('RA', 'ra')
        targets.rename_column('DEC', 'dec')
        col_formats = [".0f", "", "", "", ".3f"]
    case 3.1: # LAMMIM Cluster Sample (Kluge)
        targets_name = 'lammim_cluster_sample_redMapper_Kluge'
        targets = Table.read("../data/girmos-sci/lammim_cluster_sample_redMapper_Kluge.csv", format='csv')
        targets.rename_column('col0', 'id')
        targets.rename_column('z_cl', 'z')
        col_formats = [".0f", ".5f", ".5f", ".4f"]
    case 3.2: # LAMMIM Cluster Sample (Madcowsii)
        targets_name = 'lammim_cluster_sample_madcowsii'
        targets = Table.read("../data/girmos-sci/lammim_cluster_sample_madcowsii.csv", format='csv')
        targets.rename_column('col0', 'id')
        targets.rename_column('zspec', 'z')
        col_formats = [".0f", ".5f", ".5f", ".4f"]
    case 3.3: # LAMMIM Cluster Sample (DESI Legacy)
        targets_name = 'lammim_cluster_sample_desi_legacy_wh24'
        targets = Table.read("../data/girmos-sci/lammim_cluster_sample_desi_legacy_wh24.csv", format='csv')
        targets.rename_column('col0', 'id')
        targets.rename_column('z_cl', 'z')
        col_formats = [".0f", ".5f", ".5f", ".4f"]
    case 3.4: # LAMMIM lammim_clmem_cat_girmos_cluster_candidates
        targets_name = 'lammim_clmem_cat_girmos_cluster_candidates'
        targets = Table.read("../data/girmos-sci/lammim_clmem_cat_girmos_cluster_candidates.csv", format='csv')
        targets.rename_column('col0', 'id')
        targets.remove_column('zwarn')
        targets.remove_column('spectype')
        targets.remove_column('desi_target')
        targets.remove_column('radial_distance')
        targets.remove_column('cl_id')
        targets.remove_column('bgs_flag')
        targets.remove_column('lrg_flag')
        targets.remove_column('elg_flag')
        col_formats = [".0f", ".5f", ".5f", ".4f"]
    case 3.5: # LAMMIM lammim_orelse_clcat_girmos
        targets_name = 'lammim_orelse_clcat_girmos'
        targets = Table.read("../data/girmos-sci/lammim_orelse_clcat_girmos.csv", format='csv')
        targets.remove_column('col0')
        targets.rename_column('Id', 'id')
        targets.remove_column('e_z')
        col_formats = [".0f", ".4f", ".5f", ".5f"]
    case 3.6: # LAMMIM lammim_gal_cat_girmos_candidates
        targets_name = 'lammim_gal_cat_girmos_candidates'
        targets = Table.read("../data/girmos-sci/lammim_gal_cat_girmos_candidates.csv", format='csv')
        targets.rename_column('col0', 'id')
        targets.remove_column('zwarn')
        targets.remove_column('spectype')
        targets.remove_column('desi_target')
        targets.remove_column('bgs_flag')
        targets.remove_column('lrg_flag')
        targets.remove_column('elg_flag')
        col_formats = [".0f", ".5f", ".5f", ".4f"]
    case 4: # Hung et al for Brian
        targets_name = 'hung_et_al_2025'
        targets = Table.read("../data/girmos-sci/Hung_et_al_2025_CDS_Table3.txt", format='ascii.cds')
        targets.rename_column('CID', 'id')
        targets.rename_column('RAdeg', 'ra')
        targets.rename_column('DEdeg', 'dec')
        targets.rename_column('zsys', 'z')
        targets.remove_column('f_CID')
        targets.remove_column('Npeak')
        targets.remove_column('Amp')
        targets.remove_column('e_Amp')
        col_formats = ["", ".0f", ".5f", ".5f", ".4f"]
    case 5.1: # Katherine clumpy galaxies from 3D-HST
        targets_name = 'katherine_clumpy_galaxies_3DHST'
        targets = Table.read("../data/girmos-sci/katherine_3dHST_possible_targets_girmos.dat", format='ascii.basic')
        targets.rename_column('phot_id', 'id')
        targets.rename_column('z_best', 'z')
        remove_columns_except(targets, ['field', 'id', 'ra', 'dec', 'z'])
        col_formats = ["", ".0f", ".5f", ".5f", ".4f"]
    case 5.2: # Katherine clumpy galaxies from Euclid
        targets_name = 'katherine_clumpy_galaxies_Euclid'
        targets = Table.read("../data/girmos-sci/katherine_EuclidQ1_possible_targets_girmos.csv", format='csv')
        targets.rename_column('object_id', 'id')
        targets.rename_column('right_ascension', 'ra')
        targets.rename_column('declination', 'dec')
        targets.rename_column('spe_z', 'z')
        remove_columns_except(targets, ['id', 'ra', 'dec', 'z'])
        col_formats = [".4f", ".0f", ".5f", ".5f"]

print('Number of targets:', len(targets))

### Match

In [ ]:
def get_asterism_mags(star1_mag, star2_mag, star3_mag):
    mags = np.full(star1_mag.shape, '', dtype=object)

    mask1 = (star2_mag == -1) & (star3_mag == -1)
    if np.sum(mask1) > 0:
        mags[mask1] = [f"{m1:.1f}" for m1 in star1_mag[mask1]]

    mask2 = (star3_mag == -1) & (~mask1)
    if np.sum(mask2) > 0:
        star_mags = np.stack([star1_mag[mask2], star2_mag[mask2]], axis=1)
        star_mags_sorted = np.sort(star_mags, axis=1)
        mags[mask2] = [f"{m1:.1f},{m2:.1f}" for m1, m2 in zip(star_mags_sorted[:, 0], star_mags_sorted[:, 1])]

    mask3 = ~(mask1 | mask2)
    if np.sum(mask3) > 0:
        star_mags = np.stack([star1_mag[mask3], star2_mag[mask3], star3_mag[mask3]], axis=1)
        star_mags_sorted = np.sort(star_mags, axis=1)
        mags[mask3] = [f"{m1:.1f},{m2:.1f},{m3:.1f}" for m1, m2, m3 in zip(star_mags_sorted[:, 0], star_mags_sorted[:, 1], star_mags_sorted[:, 2])]

    return mags

In [ ]:
asterism_catalog = SkyCoord(ra=asterisms['ra'], dec=asterisms['dec'], unit='deg', frame='icrs')
if isinstance(targets['ra'][0], str) and ':' in targets['ra'][0]:
    targets_catalog = SkyCoord(ra=targets['ra'], dec=targets['dec'], unit=(u.hourangle, u.deg), frame='icrs')
else:
    targets_catalog = SkyCoord(ra=targets['ra'], dec=targets['dec'], unit=(u.deg, u.deg), frame='icrs')

idx, sep, _ = targets_catalog.match_to_catalog_sky(asterism_catalog)

if export_all:
    target_filter = np.ones(len(targets), dtype=bool)
else:
    target_filter = sep < fov/2

distance = sep[target_filter]

matches = targets[target_filter]
matches['asterism_id'] = asterisms['id'][idx][target_filter]
matches['asterism_mags'] = get_asterism_mags(asterisms['star1_mag'][idx][target_filter],asterisms['star2_mag'][idx][target_filter],asterisms['star3_mag'][idx][target_filter])
matches['asterism_ra'] = asterisms['ra'][idx][target_filter]
matches['asterism_dec'] = asterisms['dec'][idx][target_filter]
matches['asterism_distance'] = distance.to(u.arcsec).value
matches['SR_mean'] = asterisms['SR_mean'][idx][target_filter]
matches['SR_min']  = asterisms['SR_min'][idx][target_filter]
matches['SR_max']  = asterisms['SR_max'][idx][target_filter]
matches['EE100_mean'] = asterisms['EE100_mean'][idx][target_filter]
matches['EE100_min']  = asterisms['EE100_min'][idx][target_filter]
matches['EE100_max']  = asterisms['EE100_max'][idx][target_filter]
matches['FWHM_mean'] = asterisms['FWHM_mean'][idx][target_filter]
matches['FWHM_min']  = asterisms['FWHM_min'][idx][target_filter]
matches['FWHM_max']  = asterisms['FWHM_max'][idx][target_filter]

if export_all:
    matches['SR_mean'][distance >= fov/2] = np.nan
    matches['SR_min'][distance >= fov/2] = np.nan
    matches['SR_max'][distance >= fov/2] = np.nan
    matches['EE100_mean'][distance >= fov/2] = np.nan
    matches['EE100_min'][distance >= fov/2] = np.nan
    matches['EE100_max'][distance >= fov/2] = np.nan
    matches['FWHM_mean'][distance >= fov/2] = np.nan
    matches['FWHM_min'][distance >= fov/2] = np.nan
    matches['FWHM_max'][distance >= fov/2] = np.nan

col_formats = col_formats + [".0f", "", ".5f", ".5f", ".1f", ".3f", ".3f", ".3f", ".3f", ".3f", ".3f", ".1f", ".1f", ".1f"]

# Sort with NaNs at the end
if export_all:
    nan_mask = np.isnan(matches['EE100_mean'])

    valid_rows = matches[~nan_mask]
    valid_rows.sort('EE100_mean', reverse=True)

    nan_rows = matches[nan_mask]
    nan_rows.sort('asterism_distance')

    matches = vstack([valid_rows, nan_rows])
else:
    matches.sort('EE100_mean', reverse=True)

print('Number of targets near an asterism:', len(matches))
if len(matches) > 0:
    display(tabulate(matches[:200], headers=matches.colnames, tablefmt='html', floatfmt=col_formats))

### Sky Lines

In [ ]:
max_sky_line_targets = None
if 'z' in matches.colnames and (max_sky_line_targets is None or len(matches) < max_sky_line_targets):
    from survey_tools import sky
    from ao_tools import etc

    R            = 3000  # 3000 or 8000
    fov          = 2.0 * u.arcsec
    N_exp        = 24             # number of exposures
    T_exp        = 600.0 * u.s    # exposure time
    nPix         = 4              # number of pixels in aperture
    eta_spat     = 0.01           # fraction of light in aperture
    flux         = 1e-16 * u.erg/u.s/u.cm**2  # line flux to use

    # See: https://classic.sdss.org/dr6/algorithms/linestable.php
    lines = [
        {'name': 'SII'      , 'wavelength_vac': np.array([0.6732670, 0.6718290]), 'dispersion': 75.0, 'flux_ratio': np.array([0.105, 0.095]), 'snr_threshold': np.array([1.0, 1.0])},
        {'name': 'Ha'       , 'wavelength_vac': np.array([0.6564614           ]), 'dispersion': 75.0, 'flux_ratio': np.array([1.000       ]), 'snr_threshold': np.array([3.0     ])},
        {'name': 'NII'      , 'wavelength_vac': np.array([0.6585270, 0.6549860]), 'dispersion': 75.0, 'flux_ratio': np.array([0.187, 0.063]), 'snr_threshold': np.array([1.0, 1.0])},
        {'name': 'OI'       , 'wavelength_vac': np.array([0.6300304, 0.6363776]), 'dispersion': 75.0, 'flux_ratio': np.array([0.015, 0.005]), 'snr_threshold': np.array([1.0, 1.0])},
        {'name': 'Hb'       , 'wavelength_vac': np.array([0.4862721           ]), 'dispersion': 75.0, 'flux_ratio': np.array([0.350       ]), 'snr_threshold': np.array([1.0     ])},
        {'name': 'OIII_5008', 'wavelength_vac': np.array([0.5008240, 0.4960295]), 'dispersion': 75.0, 'flux_ratio': np.array([1.120, 0.380]), 'snr_threshold': np.array([3.0, 1.0])},
        {'name': 'OIII_4364', 'wavelength_vac': np.array([0.4364436           ]), 'dispersion': 75.0, 'flux_ratio': np.array([0.004       ]), 'snr_threshold': np.array([1.0     ])},
        {'name': 'OII'      , 'wavelength_vac': np.array([0.3729875, 0.3727092]), 'dispersion': 75.0, 'flux_ratio': np.array([0.262, 0.238]), 'snr_threshold': np.array([1.0, 1.0])},
        # {'name': 'MgI b1'   , 'wavelength_vac': np.array([0.5183604           ]), 'dispersion': 75.0, 'flux_ratio': np.array([1.000       ]), 'snr_threshold': np.array([1.0     ])},
        # {'name': 'CaII H'   , 'wavelength_vac': np.array([0.3969588           ]), 'dispersion': 75.0, 'flux_ratio': np.array([1.000       ]), 'snr_threshold': np.array([1.0     ])},
        # {'name': 'CaII K'   , 'wavelength_vac': np.array([0.3934777           ]), 'dispersion': 75.0, 'flux_ratio': np.array([1.000       ]), 'snr_threshold': np.array([1.0     ])},
        # {'name': 'NaI D'    , 'wavelength_vac': np.array([0.5891583, 0.5897558]), 'dispersion': 75.0, 'flux_ratio': np.array([1.000, 1.000]), 'snr_threshold': np.array([1.0, 1.0])},
    ]

    for line in lines:
        line['wavelength'] = sky.get_vacuum_to_air_wavelength(line['wavelength_vac']*u.micron).value
        matches[line['name']] = False
        col_formats.append(".0f")

    wvl_bands = etc.get_wvl_bands('GIRMOS', R)

    etc_params = {
        'fov': fov,
        'R': R,
        'bands': wvl_bands,
        'T_exp': T_exp,
        'N_exp': N_exp,
        'nPix': nPix
    }

    z = matches['z'] # (N,)
    N = z.size
    for line in lines:
        wvl_rest = line['wavelength'] * u.micron
        M = wvl_rest.size
        snr_threshold = line['snr_threshold']

        # Per-line arrays (broadcast to N×M)
        wvl_obs = wvl_rest[None, :] * (1.0 + z[:, None])
        line_flux = np.broadcast_to(flux * line['flux_ratio'], (N, M))
        sigma_v   = np.broadcast_to(line['dispersion'], (N,M))

        kept_line = np.zeros(N, dtype=bool) # default is false so lines falling between bands are rejected

        for band in wvl_bands:
            _, wvl_lo, wvl_hi = band

            band_filter = (wvl_obs >= wvl_lo) & (wvl_obs <= wvl_hi)
            row_filter = band_filter.any(axis=1)
            if not row_filter.any():
                continue

            rejects = sky.reject_emission_line(
                etc_params,
                line_flux[row_filter] * flux.unit,
                wvl_obs[row_filter],
                sigma_v[row_filter] * u.km / u.s,
                eta_spat,
                snr_threshold=snr_threshold
            )

            # Keep line if any of it's components are accepted
            kept_line[row_filter] |= ~rejects.all(axis=1)

        matches[line['name']] = kept_line

    print("     --- Emission-line rejection summary ---")
    total = len(matches)
    any_line_kept = np.zeros(total, dtype=bool)

    for line in lines:
        name = line['name']
        col  = np.asarray(matches[name], dtype=bool)
        n_ok = int(np.count_nonzero(col))
        any_line_kept |= col
        pct = (n_ok / total * 100.0) if total else 0.0
        print(f"{name:>12}: kept {n_ok:6d}/{total}  ({pct:5.1f}%)")

    n_any = int(np.count_nonzero(any_line_kept))
    pct_any = (n_any / total * 100.0) if total else 0.0
    print(f"\nTargets with ≥1 line kept: {n_any}/{total}  ({pct_any:5.1f}%)")

### Export Results

In [ ]:
matches.write(f"../output/matches-{targets_name}.csv", format='csv', overwrite=True)